# Notebook 8: Model Explainability
## Car Price Prediction Project

**Objective:** Interpret each model to understand *why* it makes predictions.

- **Linear Regression** → Coefficient interpretation
- **Decision Tree** → Node/split interpretation
- **XGBoost** → Feature importance

---

## 8.1 Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install xgboost -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

# Load data
X_train_scaled = pd.read_csv('/content/drive/MyDrive/X_train_scaled.csv')
X_test_scaled = pd.read_csv('/content/drive/MyDrive/X_test_scaled.csv')
X_train = pd.read_csv('/content/drive/MyDrive/X_train.csv')
X_test = pd.read_csv('/content/drive/MyDrive/X_test.csv')
y_train = pd.read_csv('/content/drive/MyDrive/y_train.csv').squeeze()
y_test = pd.read_csv('/content/drive/MyDrive/y_test.csv').squeeze()

# Load models
with open('/content/drive/MyDrive/linear_regression_model.pkl', 'rb') as f:
    lr_model = pickle.load(f)
with open('/content/drive/MyDrive/decision_tree_model.pkl', 'rb') as f:
    dt_model = pickle.load(f)
with open('/content/drive/MyDrive/xgboost_model.pkl', 'rb') as f:
    xgb_model = pickle.load(f)

print("Data and models loaded.")

---
## 8.2 Linear Regression — Coefficient Interpretation

Each coefficient tells us: **holding all other variables constant**, how much the price changes when this feature changes by one unit.

In [ ]:
# Get coefficients
coef_df = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(f"Intercept (baseline price): ${lr_model.intercept_:,.2f}")
print(f"\nAll Coefficients:")
coef_df

In [ ]:
# Top 20 coefficients visualization
top = coef_df.head(20)
colors = ['green' if c > 0 else 'red' for c in top['Coefficient']]

plt.figure(figsize=(10, 10))
plt.barh(range(len(top)), top['Coefficient'], color=colors)
plt.yticks(range(len(top)), top['Feature'])
plt.xlabel('Coefficient Value ($)')
plt.title('Linear Regression — Top 20 Coefficients\n(Green = increases price, Red = decreases price)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Interpreting Specific Coefficients

**Dummy variable example:** A coefficient on a dummy variable (e.g., `brand_tier_luxury`) represents the difference in price compared to the baseline category.

**Quantitative variable example:** A coefficient on a numerical feature (e.g., `horsepower`) means that for every 1-unit increase in that feature, the predicted price changes by the coefficient amount.

In [ ]:
# Identify dummy and quantitative features for interpretation
print("=" * 60)
print("INTERPRETATION EXAMPLES")
print("=" * 60)

# Find a dummy variable with high coefficient
dummy_features = [f for f in coef_df['Feature'] if '_' in f and f in X_train_scaled.columns
                  and X_train[f].nunique() == 2]
quant_features = [f for f in coef_df['Feature'] if f in X_train_scaled.columns
                  and X_train[f].nunique() > 10]

if dummy_features:
    top_dummy = coef_df[coef_df['Feature'].isin(dummy_features)].iloc[0]
    print(f"\nDummy Variable: {top_dummy['Feature']}")
    direction = 'increases' if top_dummy['Coefficient'] > 0 else 'decreases'
    print(f"  → When this feature is 1 (vs baseline), price {direction} by ${abs(top_dummy['Coefficient']):,.0f}")

if quant_features:
    top_quant = coef_df[coef_df['Feature'].isin(quant_features)].iloc[0]
    print(f"\nQuantitative Variable: {top_quant['Feature']}")
    direction = 'increases' if top_quant['Coefficient'] > 0 else 'decreases'
    print(f"  → For every 1-unit increase in {top_quant['Feature']},")
    print(f"    price {direction} by ${abs(top_quant['Coefficient']):,.0f}")

print(f"\nIMPORTANT LIMITATION:")
print(f"  These interpretations assume the data meets linear regression assumptions.")
print(f"  If data is skewed or assumptions are violated, coefficients may be unreliable.")

---
## 8.3 Decision Tree — Node Interpretation

The Decision Tree splits the data at each node based on the feature that best separates the target. **The top nodes = the most important features.**

In [ ]:
from sklearn.tree import plot_tree, export_text

# Full tree visualization (top 3 levels)
plt.figure(figsize=(28, 14))
plot_tree(dt_model, feature_names=X_train.columns, filled=True,
          rounded=True, fontsize=9, max_depth=3,
          proportion=True)
plt.title('Decision Tree — Top 3 Levels (Most Important Splits)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Text representation of the tree (top 3 levels)
tree_text = export_text(dt_model, feature_names=list(X_train.columns), max_depth=3)
print("Decision Tree Structure (Top 3 Levels):")
print(tree_text)

In [ ]:
# Interpret the root node
tree = dt_model.tree_
root_feature = X_train.columns[tree.feature[0]]
root_threshold = tree.threshold[0]

print("=" * 60)
print("DECISION TREE INTERPRETATION")
print("=" * 60)
print(f"\nRoot Node (Most Important Feature):")
print(f"  Feature:   {root_feature}")
print(f"  Threshold: {root_threshold:.2f}")
print(f"  → The first and most important split divides cars based on")
print(f"    whether {root_feature} is <= {root_threshold:.2f} or > {root_threshold:.2f}")

# Second level features
left_child = tree.children_left[0]
right_child = tree.children_right[0]

if tree.feature[left_child] >= 0:
    left_feature = X_train.columns[tree.feature[left_child]]
    left_threshold = tree.threshold[left_child]
    print(f"\nLeft Branch (low {root_feature}):")
    print(f"  Next split: {left_feature} <= {left_threshold:.2f}")

if tree.feature[right_child] >= 0:
    right_feature = X_train.columns[tree.feature[right_child]]
    right_threshold = tree.threshold[right_child]
    print(f"\nRight Branch (high {root_feature}):")
    print(f"  Next split: {right_feature} <= {right_threshold:.2f}")

In [ ]:
# Decision Tree feature importance
dt_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

top_dt = dt_imp[dt_imp['Importance'] > 0].head(15)

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_dt)), top_dt['Importance'], color='#2ecc71')
plt.yticks(range(len(top_dt)), top_dt['Feature'])
plt.xlabel('Importance')
plt.title('Decision Tree — Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 8.4 XGBoost — Feature Importance

In [ ]:
xgb_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

top_xgb = xgb_imp.head(15)

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_xgb)), top_xgb['Importance'], color='#3498db')
plt.yticks(range(len(top_xgb)), top_xgb['Feature'])
plt.xlabel('Importance')
plt.title('XGBoost — Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 10 XGBoost features:")
xgb_imp.head(10)

## 8.5 Comparison of Feature Importance Across Models

In [ ]:
# Merge importances
comp = pd.DataFrame({'Feature': X_train.columns})
comp = comp.merge(dt_imp.rename(columns={'Importance': 'DT Importance'}), on='Feature')
comp = comp.merge(xgb_imp.rename(columns={'Importance': 'XGB Importance'}), on='Feature')

# Add absolute LR coefficient (normalized)
lr_coef = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'LR |Coefficient|': np.abs(lr_model.coef_)
})
lr_coef['LR |Coefficient|'] = lr_coef['LR |Coefficient|'] / lr_coef['LR |Coefficient|'].max()
comp = comp.merge(lr_coef, on='Feature', how='left').fillna(0)

# Sort by average importance
comp['Avg Importance'] = (comp['DT Importance'] + comp['XGB Importance'] + comp['LR |Coefficient|']) / 3
comp = comp.sort_values('Avg Importance', ascending=False)

print("Feature Importance Comparison (Top 15):")
comp.head(15)

---
## Explainability Summary

| Model | Interpretation Method | Key Insight |
|-------|----------------------|-------------|
| **Linear Regression** | Coefficients | Each coefficient = change in price per unit change in feature |
| **Decision Tree** | Node splits | Top nodes show the most important features and thresholds |
| **XGBoost** | Feature importance | Ranks features by contribution to prediction accuracy |

### Key Distinction
- **Dummy variable coefficient**: Compares categories (e.g., luxury vs. economy)
- **Quantitative coefficient**: Measures incremental change (e.g., +1 HP → +$X)
- **Limitation**: Coefficients are unreliable if data is skewed or assumptions are violated

**Next step →** Notebook 09: PyCaret